In [1]:
#models
library(tidyverse)
library(caret)
library(recipes)
library(pROC)
library(dplyr)
library(tidyr)
library(ppcor)

dir.create("../../outputs/Fig3", recursive = TRUE, showWarnings = FALSE)


── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: lattice


Attaching package: ‘caret’


The following object is masked from ‘package:purrr’:

    lift



Attaching package: ‘recipes’


The following object is masked from ‘package:stringr’:

    fixed


The following object is masked from ‘package:stats’:

    step


Type 'citation("pROC")' for a citation.


Attaching packa

## Load data

In [2]:
mica_df <- read_csv("../../data/mica_df_with_pcs.csv")
lucas_df <- read_csv("../../data/lucas_df_w_pc_gemini2.csv")
val_df <- read_csv("../../data/val_df_with_pcs.csv")

mica_df <- mica_df %>%
  mutate(olink_CEA = log2(clinical_CEA + 1))

lucas_df <- lucas_df %>%
  mutate(olink_CEA = log2(clinical_CEA + 1))

mica_df_filtered <- mica_df[mica_df$`Coded Type` %in% c("Lung (LU)", "No cancer (NC)"), ]

Rows: 540 Columns: 1304
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr   (73): id, Stage, Coded Type, type, Delfi ID (Aliquot 2), DL ID, DL Suf...
dbl (1225): clinical_CCI, clinical_age, Smoking, ratio_1, ratio_2, ratio_3, ...
lgl    (6): Match?, Rack (plasma), Rack (cfDNA), check, Robot Protocol, Repl...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 287 Columns: 959
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr   (96): Patient, id, type, clinical_smokingstatus, QC, patient.type, cli...
dbl  (851): multinucratio, clinical_nlratio, clinical_CRP, clinical_cfdna_co...
lgl    (3): DIAGNOSE_OTHER_PRIMARY, Adaptor Dimer, Date Data Received
date   (9): 

## Fragmentation Features

In [3]:
lucas_df <- read_csv("../../data/lucas_df_w_pc_gemini2.csv")


lucas_df <- lucas_df %>%
  mutate(olink_CEA = log2(clinical_CEA + 1))

tcga <- readRDS("../../data/Mathios_fig2c_p2_data.rds")

df <- lucas_df

Rows: 287 Columns: 959
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr   (96): Patient, id, type, clinical_smokingstatus, QC, patient.type, cli...
dbl  (851): multinucratio, clinical_nlratio, clinical_CRP, clinical_cfdna_co...
lgl    (3): DIAGNOSE_OTHER_PRIMARY, Adaptor Dimer, Date Data Received
date   (9): DATE_1_VISIT_BBH, DATE_BIOPSY, DATE_SCAN_BASELINE, DATE_FEV1, DA...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Figure 3b/c: Copy Number Correlations

In [4]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(purrr)
  library(ComplexHeatmap)
  library(circlize)
  library(grid)
  library(ppcor)
})

# ---------------------------------------------------------------------------
# Helper: Order chromosome arms in genomic order
# ---------------------------------------------------------------------------
order_chrom_arms <- function(x) {
  x_clean <- gsub("^chr", "", x, ignore.case = TRUE)

  chr <- sub("([0-9XY]+).*", "\\1", x_clean)
  chr_num <- suppressWarnings(as.numeric(chr))
  chr_num[is.na(chr_num) & chr == "X"] <- 23
  chr_num[is.na(chr_num) & chr == "Y"] <- 24

  arm <- ifelse(grepl("p", x_clean), "p",
                ifelse(grepl("q", x_clean), "q", "z"))

  band <- suppressWarnings(as.numeric(sub(".*[pq]([0-9\\.]+).*", "\\1", x_clean)))
  band[grepl("p|q", x_clean) & is.na(band)] <- 0  

  ord <- order(chr_num, arm, band, na.last = TRUE)
  x[ord]
}

# ---------------------------------------------------------------------------
# Helper: Safe partial correlation
# ---------------------------------------------------------------------------
safe_pcor <- function(x, y, z) {
  tryCatch({
    data <- data.frame(x = x, y = y, z = z)
    data <- data[complete.cases(data), ]
    if (nrow(data) < 3) return(NA_real_)
    result <- pcor.test(data$x, data$y, data$z)
    return(result$estimate)
  }, error = function(e) {
    return(NA_real_)
  })
}

# ---------------------------------------------------------------------------
# MAIN HEATMAP FUNCTION
# ---------------------------------------------------------------------------
plot_feature_heatmap <- function(
  df,
  tcga = NULL,
  feature_type = c("ratio_pc","zscore","olink","tf","custom"),
  custom_features = NULL,
  control_for_ichor = FALSE,
  clinical_vars = c("clinical_Hgb","clinical_platelets","clinical_nlratio","clinical_survival",
                    "clinical_cfdna_conc","clinical_CRP","clinical_smokingstatus",
                    "clinical_leukocytes","clinical_age","clinical_packyears", 
                    "clinical_bmi","clinical_COPD","clinical_sex"),
  olink_groups = NULL,
  abs_cor = TRUE,
  color_scheme = c("red","blue","purple","green"),
  cor_max = 0.4
) {

  feature_type <- match.arg(feature_type)
  color_scheme <- match.arg(color_scheme)

  # ------------------------
  # Olink groups
  # ------------------------
  olink_map <- list("Tumor" = c("olink_CEA","olink_MMP12"), 
                    "Metabolic" = c("olink_ADA","olink_ARG1","olink_CAIX","olink_HO_1"),
    "Cytokines" = c("olink_IL2","olink_IL4","olink_IL5","olink_IL6","olink_IL7",
                       "olink_IL8","olink_IL10","olink_IL12","olink_IL13","olink_IL18","olink_IL33",
                       "olink_IL_1_alpha","olink_IL_21","olink_IL_35","olink_IL12RB1"),
    "T-cell" = c("olink_CD4","olink_CD5","olink_CD8A","olink_CD27","olink_CD28","olink_CRTAM"),
    "B-cell" = c("olink_CD40","olink_CD83","olink_LAMP3"),
    "NK Cell" = c("olink_NCR1","olink_KLRD1"),
    "Checkpoint" = c("olink_PD_L1","olink_PD_L2","olink_PDCD1","olink_ICOSLG"),
    "TNF Superfamily" = c("olink_TNF","olink_TNFSF14","olink_TNFRSF4","olink_TNFRSF9",
                          "olink_TNFRSF12A","olink_TNFRSF21","olink_CD40_L","olink_CD70"),
    "Apoptosis" = c("olink_TRAIL","olink_FASLG","olink_TWEAK","olink_CASP_8"),
    "Granzymes" = c("olink_GZMA","olink_GZMB","olink_GZMH"),
    "Chemokines" = c("olink_CXCL1","olink_CXCL5","olink_CXCL9","olink_CXCL10","olink_CXCL11",
                     "olink_CXCL12","olink_CXCL13","olink_CCL3","olink_CCL4","olink_CCL17",
                     "olink_CCL19","olink_CCL20","olink_CCL23","olink_MCP_1","olink_MCP_2",
                     "olink_MCP_3","olink_MCP_4","olink_CX3CL1"),
    "Angiogenesis" = c("olink_VEGFA","olink_VEGFC","olink_VEGFR_2","olink_ANG_1",
                       "olink_ANGPT2","olink_TIE2","olink_PGF","olink_NOS3"),
    "Growth Factors" = c("olink_EGF","olink_FGF2","olink_HGF","olink_PDGF_subunit_B","olink_CSF_1","olink_PTN"),
    "Interferons" = c("olink_IFN_beta","olink_IFN_gamma"),
    "Galectins" = c("olink_Gal_1","olink_Gal_9"),
    "Stress/Cytotoxicity Markers" = c("olink_MIC_A_B","olink_CD244"),
    "Other Proteins" = c("olink_DCN","olink_ADGRG1","olink_LAP_TGF_beta_1", "olink_MMP7")
  )

  # ------------------------------
  # Clinical vars + olink filtering
  # ------------------------------
  if (!is.null(olink_groups)) {
    selected_olinks <- unlist(olink_map[olink_groups])
    clinical_vars <- c(
      clinical_vars[!grepl("^olink_", clinical_vars)],
      selected_olinks
    )
  } else {
    all_olinks <- unlist(olink_map)
    clinical_vars <- c(
      clinical_vars[!grepl("^olink_", clinical_vars)],
      all_olinks
    )
  }

  # Feature columns
  feature_cols <- switch(
    feature_type,
    "ratio_pc" = grep("^ratio_pc_", names(df), value=TRUE),
    "zscore"   = grep("^zscore_",   names(df), value=TRUE),
    "olink"    = grep("^olink_",    names(df), value=TRUE),
    "tf"       = grep("^relcov-.*-rel_cov_profile$", names(df), value=TRUE),
    "custom"   = custom_features
  )
  stopifnot(length(feature_cols) > 0)

  clinical_vars <- clinical_vars[clinical_vars %in% names(df)]
  clinical_vars <- clinical_vars[sapply(df[clinical_vars], is.numeric)]

  if (!control_for_ichor && "ichor" %in% names(df)) {
    clinical_vars <- c(clinical_vars, "ichor")
  }

  # ---------------------------------
  # Compute correlations
  # ---------------------------------
  cor_long <- expand.grid(
    clinical = clinical_vars,
    feature  = feature_cols,
    stringsAsFactors = FALSE
  ) %>%
    mutate(cor = pmap_dbl(list(clinical, feature), function(cvar, fvar) {
      x <- df[[cvar]]
      y <- df[[fvar]]
      if (control_for_ichor) {
        safe_pcor(x, y, df$ichor)
      } else {
        suppressWarnings(cor(x, y, use="pairwise.complete.obs"))
      }
    }))

  if (abs_cor) cor_long$cor <- abs(cor_long$cor)

  cor_mat <- cor_long %>%
    pivot_wider(names_from = feature, values_from = cor) %>%
    column_to_rownames("clinical") %>%
    as.matrix()

  cor_mat[is.na(cor_mat)] <- 0

  # ---------------------------------
  # Row grouping
  # ---------------------------------
  original_rownames <- rownames(cor_mat)

  row_split <- sapply(original_rownames, function(var) {
    if (var %in% c("ichor", "GEMINI_score", "DELFI-TS")) {
      "01_ctDNA"
    } else if (grepl("^clinical_leukocytes$|^clinical_nlratio$", var)) {
      "02_Inflammatory Markers"
    } else if (grepl("^clinical_Hgb$|^clinical_platelets$", var)) {
      "03_Hematologic"
    } else if (grepl("^clinical_age$|^clinical_sex$|^clinical_bmi$", var)) {
      "04_Demographics"
    } else if (grepl("^clinical_smokingstatus$|^clinical_packyears$|^clinical_COPD$", var)) {
      "05_Smoking/Respiratory"
    } else if (grepl("^clinical_CRP$|^clinical_cfdna_conc$", var)) {
      "06_cfDNA/CRP"
    } else if (grepl("^clinical_survival$", var)) {
      "99_Survival"
    } else if (grepl("^clinical_", var)) {
      "07_Other Clinical"
    } else if (grepl("^olink_", var)) {
      for (group_name in names(olink_map)) {
        if (var %in% olink_map[[group_name]]) {
          group_num <- match(group_name, names(olink_map)) + 9
          return(sprintf("%02d_%s", group_num, group_name))
        }
      }
      "89_Other Proteins"
    } else {
      "07_Other Clinical"
    }
  })

  row_info <- data.frame(
    original_name = original_rownames,
    category = row_split,
    stringsAsFactors = FALSE
  )
  row_info <- row_info[order(row_info$category), ]

  cor_mat <- cor_mat[row_info$original_name, , drop = FALSE]
  row_split <- row_info$category

  # ---------------------------------
  # Clean labels
  # ---------------------------------
  rownames(cor_mat) <- gsub("^clinical_", "", rownames(cor_mat))
  rownames(cor_mat) <- tools::toTitleCase(rownames(cor_mat))
  rownames(cor_mat) <- gsub("^Ichor$", "ichorCNA        ", rownames(cor_mat))
  rownames(cor_mat) <- gsub("GEMINI_score", "GEMINI        ", rownames(cor_mat))
  rownames(cor_mat) <- gsub("Cfdna_conc", "cfDNA Conc.", rownames(cor_mat))
  rownames(cor_mat) <- gsub("Nlratio", "Neutrophiles", rownames(cor_mat))
  rownames(cor_mat) <- gsub("^olink_", "", rownames(cor_mat))

  colnames(cor_mat) <- gsub("^ratio_pc_0*", "", colnames(cor_mat))
  colnames(cor_mat) <- gsub("^zscore_", "", colnames(cor_mat))

  # ---------------------------------
  # Order columns genomically
  # ---------------------------------
  ordered_cols <- order_chrom_arms(colnames(cor_mat))
  cor_mat <- cor_mat[, ordered_cols, drop = FALSE]

  # ===================================================================
  # TCGA TITLE PANEL (placeholder for decoration)
  # ===================================================================
  tcga_title_panel <- NULL
  if (!is.null(tcga)) {
    tcga_title_panel <- HeatmapAnnotation(
      tcga_title = anno_empty(border = FALSE, height = unit(1.5, "cm")),
      annotation_name_side = "left",
      annotation_name_gp = gpar(fontsize = 0),
      show_annotation_name = FALSE
    )
  }

  # ===================================================================
  # TOP PANELS - TCGA GAINS/LOSSES (SEPARATE FOR LUAD AND LUSC)
  # ===================================================================
  tcga_luad_panel <- NULL
  tcga_lusc_panel <- NULL
  
  if (!is.null(tcga)) {
    # Get unique arms in order matching the heatmap
    heatmap_arms <- colnames(cor_mat)
    
    # Separate gains and losses, then aggregate by arm and disease
    tcga_summary <- tcga %>%
      mutate(arm_label = as.character(arm)) %>%
      mutate(category = ifelse(value > 0, "Gain", "Loss")) %>%
      group_by(arm_label, disease, category) %>%
      summarise(median_value = median(value, na.rm = TRUE), .groups = "drop")
    
    # Create matched data frame for heatmap columns
    tcga_matched <- data.frame(
      arm = heatmap_arms,
      LUAD_gain = 0,
      LUAD_loss = 0,
      LUSC_gain = 0,
      LUSC_loss = 0,
      stringsAsFactors = FALSE
    )
    
    # Fill in values from tcga_summary
    for (i in 1:nrow(tcga_matched)) {
      arm <- tcga_matched$arm[i]
      
      # LUAD gains
      luad_gain <- tcga_summary %>% 
        filter(arm_label == arm, disease == "LUAD", category == "Gain") %>% 
        pull(median_value)
      if (length(luad_gain) > 0) tcga_matched$LUAD_gain[i] <- luad_gain[1]
      
      # LUAD losses (store as negative)
      luad_loss <- tcga_summary %>% 
        filter(arm_label == arm, disease == "LUAD", category == "Loss") %>% 
        pull(median_value)
      if (length(luad_loss) > 0) tcga_matched$LUAD_loss[i] <- luad_loss[1]
      
      # LUSC gains
      lusc_gain <- tcga_summary %>% 
        filter(arm_label == arm, disease == "LUSC", category == "Gain") %>% 
        pull(median_value)
      if (length(lusc_gain) > 0) tcga_matched$LUSC_gain[i] <- lusc_gain[1]
      
      # LUSC losses (store as negative)
      lusc_loss <- tcga_summary %>% 
        filter(arm_label == arm, disease == "LUSC", category == "Loss") %>% 
        pull(median_value)
      if (length(lusc_loss) > 0) tcga_matched$LUSC_loss[i] <- lusc_loss[1]
    }
    
    # Determine y-axis scale based on data range
    y_max <- max(abs(c(tcga_matched$LUAD_gain, tcga_matched$LUAD_loss, 
                        tcga_matched$LUSC_gain, tcga_matched$LUSC_loss)), na.rm = TRUE)
    y_max <- ceiling(y_max * 10) / 10  # Round up to nearest 0.1
    y_range <- c(-y_max, y_max)
    
    # Create annotation function for LUAD
    anno_tcga_luad <- AnnotationFunction(
      fun = function(index, k, n) {
        n_items <- length(index)
        pushViewport(viewport(xscale = c(0.5, n_items + 0.5), yscale = y_range))
        
        # Draw background rectangle
        grid.rect(gp = gpar(fill = NA, col = "black", lwd = 2.5))
        
        # Draw horizontal line at y=0
        grid.segments(0.5, 0, n_items + 0.5, 0, 
                     default.units = "native", 
                     gp = gpar(col = "grey50", lty = 2, lwd = 1.5))
        
        # Draw bars for LUAD (gains above 0, losses below 0)
        bar_width <- 0.7
        for (i in 1:n_items) {
          idx <- index[i]
          gain_val <- tcga_matched$LUAD_gain[idx]
          loss_val <- tcga_matched$LUAD_loss[idx]
          
          # Gain bar (above 0, red)
          if (!is.na(gain_val) && gain_val > 0) {
            grid.rect(
              x = i,
              y = 0,
              width = bar_width,
              height = gain_val,
              just = c("center", "bottom"),
              default.units = "native",
              gp = gpar(fill = "#E41A1C", col = "black", lwd = 1.5)
            )
          }
          
          # Loss bar (below 0, blue) - loss_val should already be negative
          if (!is.na(loss_val) && loss_val < 0) {
            grid.rect(
              x = i,
              y = 0,
              width = bar_width,
              height = abs(loss_val),
              just = c("center", "top"),
              default.units = "native",
              gp = gpar(fill = "#377EB8", col = "black", lwd = 1.5)
            )
          }
        }
        
        # Draw y-axis on first slice
        if (k == 1) {
          grid.yaxis(gp = gpar(fontsize = 20, lwd = 2.5))
        }
        
        popViewport()
      },
      var_import = list(tcga_matched = tcga_matched, y_range = y_range),
      n = nrow(tcga_matched),
      subsettable = TRUE,
      height = unit(6, "cm")
    )
    
    # Create annotation function for LUSC
    anno_tcga_lusc <- AnnotationFunction(
      fun = function(index, k, n) {
        n_items <- length(index)
        pushViewport(viewport(xscale = c(0.5, n_items + 0.5), yscale = y_range))
        
        # Draw background rectangle
        grid.rect(gp = gpar(fill = NA, col = "black", lwd = 2.5))
        
        # Draw horizontal line at y=0
        grid.segments(0.5, 0, n_items + 0.5, 0, 
                     default.units = "native", 
                     gp = gpar(col = "grey50", lty = 2, lwd = 1.5))
        
        # Draw bars for LUSC (gains above 0, losses below 0)
        bar_width <- 0.7
        for (i in 1:n_items) {
          idx <- index[i]
          gain_val <- tcga_matched$LUSC_gain[idx]
          loss_val <- tcga_matched$LUSC_loss[idx]
          
          # Gain bar (above 0, red)
          if (!is.na(gain_val) && gain_val > 0) {
            grid.rect(
              x = i,
              y = 0,
              width = bar_width,
              height = gain_val,
              just = c("center", "bottom"),
              default.units = "native",
              gp = gpar(fill = "#E41A1C", col = "black", lwd = 1.5)
            )
          }
          
          # Loss bar (below 0, blue) - loss_val should already be negative
          if (!is.na(loss_val) && loss_val < 0) {
            grid.rect(
              x = i,
              y = 0,
              width = bar_width,
              height = abs(loss_val),
              just = c("center", "top"),
              default.units = "native",
              gp = gpar(fill = "#377EB8", col = "black", lwd = 1.5)
            )
          }
        }
        
        # Draw y-axis on first slice
        if (k == 1) {
          grid.yaxis(gp = gpar(fontsize = 20, lwd = 2.5))
        }
        
        popViewport()
      },
      var_import = list(tcga_matched = tcga_matched, y_range = y_range),
      n = nrow(tcga_matched),
      subsettable = TRUE,
      height = unit(6, "cm")
    )
    
    tcga_luad_panel <- HeatmapAnnotation(
      `LUAD\n\n` = anno_tcga_luad,
      annotation_name_side = "left",
      annotation_name_gp   = gpar(fontsize = 30),
      annotation_name_rot  = 90,
      gap = unit(2, "mm")
    )
    
    tcga_lusc_panel <- HeatmapAnnotation(
      `LUSC\n\n` = anno_tcga_lusc,
      annotation_name_side = "left",
      annotation_name_gp   = gpar(fontsize = 30),
      annotation_name_rot  = 90,
      gap = unit(2, "mm")
    )
  }

  # ===================================================================
  # SPACER PANELS - Each with unique name
  # ===================================================================
  spacer_panel_1 <- HeatmapAnnotation(
    spacer1 = anno_empty(border = FALSE, height = unit(1, "cm")),
    show_annotation_name = FALSE
  )
  
  spacer_panel_2 <- HeatmapAnnotation(
    spacer2 = anno_empty(border = FALSE, height = unit(1, "cm")),
    show_annotation_name = FALSE
  )
  
  spacer_panel_3 <- HeatmapAnnotation(
    spacer3 = anno_empty(border = FALSE, height = unit(1, "cm")),
    show_annotation_name = FALSE
  )

  # ===================================================================
  # LUCAS COHORT TITLE PANEL (placeholder for decoration)
  # ===================================================================
  lucas_title_panel <- NULL
  if (feature_type == "zscore") {
    lucas_title_panel <- HeatmapAnnotation(
      lucas_title = anno_empty(border = FALSE, height = unit(1.5, "cm")),
      annotation_name_side = "left",
      annotation_name_gp = gpar(fontsize = 0),
      show_annotation_name = FALSE
    )
  }

  # ===================================================================
  # MIDDLE PANEL 1 - CANCER PATIENTS (individual patient z-score points)
  # ===================================================================
  zscore_cancer_panel <- NULL

  if (feature_type == "zscore") {

    zscore_cols <- grep("^zscore_", names(df), value = TRUE)
    df_cancer <- df[df$type == "cancer", , drop = FALSE]

    if (nrow(df_cancer) > 0 && length(zscore_cols) > 0) {

      z_raw <- df_cancer[, zscore_cols, drop = FALSE]
      z_raw <- as.data.frame(lapply(z_raw, as.numeric))
      colnames(z_raw) <- gsub("^zscore_", "", zscore_cols)

      missing_cols <- setdiff(colnames(cor_mat), colnames(z_raw))
      if (length(missing_cols) > 0) {
        for (mc in missing_cols) z_raw[[mc]] <- NA_real_
      }

      z_raw <- z_raw[, colnames(cor_mat), drop = FALSE]

      # Prepare data for plotting individual points
      z_long <- z_raw %>%
        mutate(patient_id = row_number()) %>%
        pivot_longer(cols = -patient_id, names_to = "arm", values_to = "zscore") %>%
        filter(!is.na(zscore))
      
      # Create a mapping of arm names to column indices
      arm_indices <- data.frame(
        arm = colnames(z_raw),
        index = 1:ncol(z_raw),
        stringsAsFactors = FALSE
      )
      
      z_long <- z_long %>%
        left_join(arm_indices, by = "arm") %>%
        as.data.frame()

      # Create annotation function for individual patient points
      anno_patient_points <- AnnotationFunction(
        fun = function(index, k, n) {
          n_items <- length(index)
          pushViewport(viewport(xscale = c(0.5, n_items + 0.5), yscale = c(-100, 100)))
          
          # Draw background rectangle
          grid.rect(gp = gpar(fill = NA, col = "black", lwd = 2.5))
          
          # Draw horizontal line at y=0
          grid.segments(0.5, 0, n_items + 0.5, 0, 
                       default.units = "native", 
                       gp = gpar(col = "grey50", lty = 2, lwd = 1.5))
          
          # Draw individual patient points
          for (i in 1:n_items) {
            idx <- index[i]
            
            arm_data <- z_long[z_long$index == idx, , drop = FALSE]
            
            if (nrow(arm_data) > 0) {
              z_values <- arm_data$zscore
              
              for (j in 1:length(z_values)) {
                z_val <- z_values[j]
                
                # Clip to range
                z_val_clipped <- max(-90, min(90, z_val))
                
                # Color based on gain (positive) or loss (negative)
                point_color <- ifelse(z_val > 0, "#E41A1C", "#377EB8")
                
                # Add small jitter to x position
                jitter_amount <- 0.15
                x_pos <- i + runif(1, -jitter_amount, jitter_amount)
                
                # Draw point
                grid.points(
                  x = x_pos,
                  y = z_val_clipped,
                  default.units = "native",
                  pch = 21,
                  gp = gpar(fill = point_color, col = "black", lwd = 1),
                  size = unit(3, "mm")
                )
              }
            }
          }
          
          # Draw y-axis on first slice
          if (k == 1) {
            grid.yaxis(gp = gpar(fontsize = 25, lwd = 2.5))
          }
          
          popViewport()
        },
        var_import = list(z_long = z_long),
        n = ncol(z_raw),
        subsettable = TRUE,
        height = unit(12, "cm")
      )

      zscore_cancer_panel <- HeatmapAnnotation(
        `Cancer Patient\nZ-scores\n\n` = anno_patient_points,
        annotation_name_side = "left",
        annotation_name_gp   = gpar(fontsize = 30),
        annotation_name_rot  = 90,
        gap = unit(2, "mm")
      )
    }
  }
  
  # ===================================================================
  # MIDDLE PANEL 2 - HEALTHY PATIENTS (individual patient z-score points in grey)
  # ===================================================================
  zscore_healthy_panel <- NULL

  if (feature_type == "zscore") {

    zscore_cols <- grep("^zscore_", names(df), value = TRUE)
    df_healthy <- df[df$type == "healthy", , drop = FALSE]

    if (nrow(df_healthy) > 0 && length(zscore_cols) > 0) {

      z_raw_healthy <- df_healthy[, zscore_cols, drop = FALSE]
      z_raw_healthy <- as.data.frame(lapply(z_raw_healthy, as.numeric))
      colnames(z_raw_healthy) <- gsub("^zscore_", "", zscore_cols)

      missing_cols <- setdiff(colnames(cor_mat), colnames(z_raw_healthy))
      if (length(missing_cols) > 0) {
        for (mc in missing_cols) z_raw_healthy[[mc]] <- NA_real_
      }

      z_raw_healthy <- z_raw_healthy[, colnames(cor_mat), drop = FALSE]

      # Prepare data for plotting individual points
      z_long_healthy <- z_raw_healthy %>%
        mutate(patient_id = row_number()) %>%
        pivot_longer(cols = -patient_id, names_to = "arm", values_to = "zscore") %>%
        filter(!is.na(zscore))
      
      # Create a mapping of arm names to column indices
      arm_indices <- data.frame(
        arm = colnames(z_raw_healthy),
        index = 1:ncol(z_raw_healthy),
        stringsAsFactors = FALSE
      )
      
      z_long_healthy <- z_long_healthy %>%
        left_join(arm_indices, by = "arm") %>%
        as.data.frame()

      # Create annotation function for individual healthy patient points (GREY)
      anno_healthy_points <- AnnotationFunction(
        fun = function(index, k, n) {
          n_items <- length(index)
          pushViewport(viewport(xscale = c(0.5, n_items + 0.5), yscale = c(-100, 100)))
          
          # Draw background rectangle
          grid.rect(gp = gpar(fill = NA, col = "black", lwd = 2.5))
          
          # Draw horizontal line at y=0
          grid.segments(0.5, 0, n_items + 0.5, 0, 
                       default.units = "native", 
                       gp = gpar(col = "grey50", lty = 2, lwd = 1.5))
          
          # Draw individual patient points in GREY
          for (i in 1:n_items) {
            idx <- index[i]
            
            arm_data <- z_long_healthy[z_long_healthy$index == idx, , drop = FALSE]
            
            if (nrow(arm_data) > 0) {
              z_values <- arm_data$zscore
              
              for (j in 1:length(z_values)) {
                z_val <- z_values[j]
                
                # Clip to range
                z_val_clipped <- max(-90, min(90, z_val))
                
                # All points in grey
                point_color <- "grey60"
                
                # Add small jitter to x position
                jitter_amount <- 0.15
                x_pos <- i + runif(1, -jitter_amount, jitter_amount)
                
                # Draw point
                grid.points(
                  x = x_pos,
                  y = z_val_clipped,
                  default.units = "native",
                  pch = 21,
                  gp = gpar(fill = point_color, col = "black", lwd = 1),
                  size = unit(3, "mm")
                )
              }
            }
          }
          
          # Draw y-axis on first slice
          if (k == 1) {
            grid.yaxis(gp = gpar(fontsize = 25, lwd = 2.5))
          }
          
          popViewport()
        },
        var_import = list(z_long_healthy = z_long_healthy),
        n = ncol(z_raw_healthy),
        subsettable = TRUE,
        height = unit(12, "cm")
      )

      zscore_healthy_panel <- HeatmapAnnotation(
        `Non-Cancer Individual\nZ-scores\n\n` = anno_healthy_points,
        annotation_name_side = "left",
        annotation_name_gp   = gpar(fontsize = 30),
        annotation_name_rot  = 90,
        gap = unit(2, "mm")
      )
    }
  }
  
  # ===================================================================
  # Combine top annotations in order
  # ===================================================================
  top_panel <- NULL
  annotation_list <- list()
  
  # Add TCGA title first
  if (!is.null(tcga_title_panel)) {
    annotation_list <- c(annotation_list, list(tcga_title_panel))
  }
  
  if (!is.null(tcga_luad_panel)) {
    annotation_list <- c(annotation_list, list(tcga_luad_panel))
  }
  if (!is.null(tcga_lusc_panel)) {
    annotation_list <- c(annotation_list, list(tcga_lusc_panel))
  }
  
  # Add first spacer if we have TCGA panels
  if (!is.null(tcga_luad_panel) || !is.null(tcga_lusc_panel)) {
    annotation_list <- c(annotation_list, list(spacer_panel_1))
  }
  
  # Add LUCAS Cohort title before z-score panels
  if (!is.null(lucas_title_panel)) {
    annotation_list <- c(annotation_list, list(lucas_title_panel))
  }
  
  if (!is.null(zscore_cancer_panel)) {
    annotation_list <- c(annotation_list, list(zscore_cancer_panel))
  }
  
  # Add spacer between cancer and healthy panels if both exist
  if (!is.null(zscore_cancer_panel) && !is.null(zscore_healthy_panel)) {
    annotation_list <- c(annotation_list, list(spacer_panel_2))
  }
  
  if (!is.null(zscore_healthy_panel)) {
    annotation_list <- c(annotation_list, list(zscore_healthy_panel))
  }
  
  # Add third spacer if we have z-score panels
  if (!is.null(zscore_cancer_panel) || !is.null(zscore_healthy_panel)) {
    annotation_list <- c(annotation_list, list(spacer_panel_3))
  }
  
  if (length(annotation_list) > 0) {
    top_panel <- do.call(c, annotation_list)
  }

  # ---------------------------------
  # Color scale
  # ---------------------------------
  mid_point <- cor_max / 2
  col_fun <- switch(
    color_scheme,
    "red"    = colorRamp2(c(0, mid_point, cor_max), c("white","orange","red")),
    "blue"   = colorRamp2(c(0, mid_point, cor_max), c("white","lightblue","darkblue")),
    "purple" = colorRamp2(c(0, mid_point, cor_max), c("white","plum","purple4")),
    "green"  = colorRamp2(c(0, mid_point, cor_max), c("white","lightgreen","darkgreen"))
  )

  legend_breaks <- seq(0, cor_max, length.out = 5)
  legend_labels <- sprintf("%.2f", legend_breaks)

  row_split <- factor(row_split, levels = unique(row_split))

  # ---------------------------------
  # Create clean group labels with padding for spacing
  # ---------------------------------
  clean_group_labels <- gsub("^[0-9]+_", "", levels(row_split))
  clean_group_labels <- paste0("     ", clean_group_labels)

  # ---------------------------------
  # HEATMAP
  # ---------------------------------
  column_title_label <- if (feature_type == "zscore") "Z-Score CN-Arms" else "Fragmentome Ratio PC"

  ht <- Heatmap(
    cor_mat,
    top_annotation = top_panel,
    name = if (control_for_ichor) "Partial Correlation" else "Correlation",
    col = col_fun,
    cluster_rows = TRUE,
    cluster_row_slices = FALSE,
    cluster_columns = if (feature_type == "zscore") FALSE else TRUE,
    show_column_dend = if (feature_type == "zscore") FALSE else TRUE,
    column_dend_side = "top",
    column_dend_height = unit(3, "cm"),
    row_split = row_split,
    row_gap = unit(5, "mm"),
    
    # Group labels on the right side
    row_title = clean_group_labels,
    row_title_side = "left", 
    row_title_rot = 0,
    row_title_gp = gpar(fontsize = 26), ### change

    column_title = column_title_label,
    column_title_side = "bottom",
    column_title_gp = gpar(fontsize = 24),

    row_names_side = "right",
    column_names_side = "bottom",
    row_names_gp = gpar(fontsize = 20), ##### change
    column_names_gp = gpar(fontsize = 20),
    column_names_rot = 45,
    column_names_max_height = unit(10, "cm"),

    rect_gp = gpar(col = "black", lwd = 0.5),
    width = unit(ncol(cor_mat), "cm"),
    height = unit(nrow(cor_mat), "cm"),

    heatmap_legend_param = list(
      title = if (control_for_ichor) "Partial Corr." else "|Corr.|",
      legend_direction = "vertical",
      title_position = "leftcenter-rot",
      at = legend_breaks,
      labels = legend_labels,
      border = "black",
      legend_gp = gpar(col = "black", lwd = 2),
      legend_height = unit(5, "cm"),
      grid_height = unit(12, "mm"),
      grid_width = unit(6, "mm"),
      labels_gp = gpar(fontsize = 15),
      title_gp = gpar(fontsize = 20)
    )
  )

  options(repr.plot.width = 25, repr.plot.height = 50, repr.plot.res = 300)
  draw(ht, heatmap_legend_side = "left", padding = unit(c(10,2,10,2),"mm"))
  
  # ===================================================================
  # Add titles via decoration after drawing
  # ===================================================================
  if (!is.null(tcga)) {
    decorate_annotation("tcga_title", {
      grid.text("TCGA", x = unit(0.5, "npc"), y = unit(0.5, "npc"),
                gp = gpar(fontsize = 34))
    })
  }
  
  if (feature_type == "zscore") {
    decorate_annotation("lucas_title", {
      grid.text("\nLUCAS Cohort\n", x = unit(0.5, "npc"), y = unit(0.5, "npc"),
                gp = gpar(fontsize = 34))
    })
  }
}

In [5]:
# Extract the correlation matrix data from the function
save_heatmap_data <- function(
  df,
  tcga = NULL,
  feature_type = c("ratio_pc","zscore","olink","tf","custom"),
  custom_features = NULL,
  control_for_ichor = FALSE,
  clinical_vars = c("clinical_Hgb","clinical_platelets","clinical_nlratio","clinical_survival",
                    "clinical_cfdna_conc","clinical_CRP","clinical_smokingstatus",
                    "clinical_leukocytes","clinical_age","clinical_packyears", 
                    "clinical_bmi","clinical_COPD","clinical_sex"),
  olink_groups = NULL
) {
  
  feature_type <- match.arg(feature_type)
  
  # Olink groups mapping
 olink_map <- list(
    "Tumor" = c("olink_CEA","olink_MMP12"), 
    "Metabolic" = c("olink_ADA","olink_ARG1","olink_CAIX","olink_HO_1"),
    "Cytokines" = c("olink_IL2","olink_IL4","olink_IL5","olink_IL6","olink_IL7",
                       "olink_IL8","olink_IL10","olink_IL12","olink_IL13","olink_IL18","olink_IL33",
                       "olink_IL_1_alpha","olink_IL_21","olink_IL_35","olink_IL12RB1"),
    "T-cell" = c("olink_CD4","olink_CD5","olink_CD8A","olink_CD27","olink_CD28","olink_CRTAM"),
    "B-cell" = c("olink_CD40","olink_CD83","olink_LAMP3"),
    "NK Cell" = c("olink_NCR1","olink_KLRD1"),
    "Checkpoint" = c("olink_PD_L1","olink_PD_L2","olink_PDCD1","olink_ICOSLG"),
    "TNF Superfamily" = c("olink_TNF","olink_TNFSF14","olink_TNFRSF4","olink_TNFRSF9",
                          "olink_TNFRSF12A","olink_TNFRSF21","olink_CD40_L","olink_CD70"),
    "Apoptosis" = c("olink_TRAIL","olink_FASLG","olink_TWEAK","olink_CASP_8"),
    "Granzymes" = c("olink_GZMA","olink_GZMB","olink_GZMH"),
    "Chemokines" = c("olink_CXCL1","olink_CXCL5","olink_CXCL9","olink_CXCL10","olink_CXCL11",
                     "olink_CXCL12","olink_CXCL13","olink_CCL3","olink_CCL4","olink_CCL17",
                     "olink_CCL19","olink_CCL20","olink_CCL23","olink_MCP_1","olink_MCP_2",
                     "olink_MCP_3","olink_MCP_4","olink_CX3CL1"),
    "Angiogenesis" = c("olink_VEGFA","olink_VEGFC","olink_VEGFR_2","olink_ANG_1",
                       "olink_ANGPT2","olink_TIE2","olink_PGF","olink_NOS3"),
    "Growth Factors" = c("olink_EGF","olink_FGF2","olink_HGF","olink_PDGF_subunit_B","olink_CSF_1","olink_PTN"),
    "Interferons" = c("olink_IFN_beta","olink_IFN_gamma"),
    "Galectins" = c("olink_Gal_1","olink_Gal_9"),
    "Stress/Cytotoxicity Markers" = c("olink_MIC_A_B","olink_CD244"),
    "Other Proteins" = c("olink_DCN","olink_ADGRG1","olink_LAP_TGF_beta_1","olink_MMP7")
  )
  
  # Filter olink vars
  if (!is.null(olink_groups)) {
    selected_olinks <- unlist(olink_map[olink_groups])
    clinical_vars <- c(clinical_vars[!grepl("^olink_", clinical_vars)], selected_olinks)
  } else {
    clinical_vars <- c(clinical_vars[!grepl("^olink_", clinical_vars)], unlist(olink_map))
  }
  
  # Feature columns
  feature_cols <- switch(
    feature_type,
    "zscore" = grep("^zscore_", names(df), value = TRUE),
    "ratio_pc" = grep("^ratio_pc_", names(df), value = TRUE),
    "olink" = grep("^olink_", names(df), value = TRUE),
    "tf" = grep("^relcov-.*-rel_cov_profile$", names(df), value = TRUE),
    "custom" = custom_features
  )
  
  clinical_vars <- clinical_vars[clinical_vars %in% names(df)]
  clinical_vars <- clinical_vars[sapply(df[clinical_vars], is.numeric)]
  
  if (!control_for_ichor && "ichor" %in% names(df)) {
    clinical_vars <- c(clinical_vars, "ichor")
  }
  
  # Compute correlations
  cor_long <- expand.grid(clinical = clinical_vars, feature = feature_cols, stringsAsFactors = FALSE) %>%
    mutate(cor = purrr::pmap_dbl(list(clinical, feature), function(cvar, fvar) {
      suppressWarnings(cor(df[[cvar]], df[[fvar]], use = "pairwise.complete.obs"))
    }))
  
  cor_mat <- cor_long %>%
    pivot_wider(names_from = feature, values_from = cor) %>%
    column_to_rownames("clinical")
  
  list(
    cor_matrix = cor_mat,
    cor_long = cor_long
  )
}

# Use it
heatmap_data <- save_heatmap_data(
  df,
  feature_type = "zscore",
  tcga = tcga,
  clinical_vars = c("GEMINI_score"),
  olink_groups = c("Tumor", "T-cell", "B-cell", "NK Cell",
                   "Checkpoint", "Granzymes", "Angiogenesis", "Apoptosis", "Cytokines")
)

heatmap_data$cor_matrix

# Save
write.csv(heatmap_data$cor_matrix, "../../data/Zscore_heatmap_correlation_matrix.csv")

,zscore_1p,zscore_1q,zscore_2p,zscore_2q,zscore_3p,zscore_3q,zscore_4p,zscore_4q,zscore_5p,zscore_5q,⋯,zscore_17p,zscore_17q,zscore_18p,zscore_18q,zscore_19p,zscore_19q,zscore_20p,zscore_20q,zscore_21q,zscore_22q
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
GEMINI_score,-0.0117032318,0.198537429,-0.039878104,-0.101103963,-0.33735436,0.274876210,-0.164244124,-0.235616834,0.326150398,-0.264310504,⋯,-0.227945871,0.2137860831,0.167950147,0.1943523780,0.0862760977,0.2431410018,9.636082e-02,0.240181798,-0.0267589051,-0.254531946
olink_CEA,-0.1337643557,0.089933004,-0.194967770,-0.056788807,-0.43175012,0.140577833,-0.077042079,-0.123237159,0.501385615,-0.291754789,⋯,-0.108527171,0.3936976655,0.039006413,0.0948364979,0.0360949478,0.2190386593,-5.612482e-04,0.256646030,-0.3027978413,-0.317837841
olink_MMP12,-0.0465520150,0.339630634,-0.244087263,-0.070609710,-0.43483493,0.344768587,-0.184543397,-0.198701030,0.505619508,-0.354604382,⋯,-0.234685033,0.3614327861,0.244569128,0.2606073714,0.1242008711,0.4118108095,1.041152e-01,0.341806251,-0.0552403636,-0.351149031
olink_CD4,-0.0432266658,0.071147478,-0.103539721,0.037743643,-0.13879465,0.096467112,0.026361740,0.039023096,0.235374647,-0.129581478,⋯,0.014992147,0.0883526759,0.141514185,0.1043522948,-0.0070227319,0.1121501265,5.804174e-02,0.059361547,-0.1374989981,-0.055292352
olink_CD5,-0.0706653119,0.018398046,-0.088077838,0.088400506,-0.06305269,-0.002504702,0.099419966,0.112639831,0.120253830,-0.090672297,⋯,0.113870162,0.0669975765,0.121548277,0.1158798396,0.0380944449,0.0184943021,8.607401e-02,-0.076314557,-0.0629860266,0.005297696
olink_CD8A,-0.0507579203,0.040322646,0.004047231,0.115794159,-0.07493632,-0.004247730,0.075681011,0.103327484,0.089976689,-0.053996145,⋯,0.109935699,0.0840970024,-0.005940033,0.0123925046,0.0774084536,0.0171331902,6.216574e-02,-0.041635339,-0.0779689315,0.025088078
olink_CD27,-0.0631519418,0.038520384,-0.098746590,0.054848186,-0.05366419,0.074831854,0.054741344,0.063811727,0.148383385,-0.065540447,⋯,0.064495729,0.0853861422,0.110437420,0.0625593388,-0.0000700132,0.0557316197,4.672721e-02,0.018584108,-0.0208043049,-0.015050627
olink_CD28,0.0348575902,0.015871645,0.026484625,0.169425816,-0.02120066,0.032157108,0.130113519,0.097522027,0.046942120,-0.057845787,⋯,0.054230524,-0.0355708527,0.119200131,0.0329212033,0.0389196761,-0.0181731384,1.825107e-01,-0.067887274,0.0162837998,0.015772672
olink_CRTAM,0.1018765773,0.005269756,0.042418780,0.091034647,-0.02034076,0.050834067,0.131856564,0.105239636,0.026666139,-0.018756732,⋯,0.108931691,-0.0426794577,0.052361576,0.0360617567,0.0374247667,0.0259667309,1.340761e-01,-0.066387527,0.0621894923,0.085325017


In [6]:
plot_feature_heatmap(
  df,
  feature_type = "zscore",
  color_scheme = "red",
  cor_max = 0.5,
  tcga = tcga,
  clinical_vars = c("GEMINI_score"),
  olink_groups = c(
    "Tumor", "T-cell", "B-cell", "NK Cell",
    "Checkpoint", "Granzymes", 
    "Angiogenesis", "Apoptosis", "Cytokines"
  )
)

In [7]:
dir.create("../../outputs/Fig3", recursive = TRUE, showWarnings = FALSE)

png("../../outputs/Fig3/Fig-Zscore_scatter_heatmap.png", width = 25, height = 50, units = "in", res = 600)

plot_feature_heatmap(
  df,
  feature_type = "zscore",
  color_scheme = "red",
  cor_max = 0.5,
    tcga = tcga,
    clinical_vars=c("GEMINI_score"),
  olink_groups = c(
    "Tumor", "T-cell","B-cell", "NK Cell",
      "Checkpoint", "Granzymes", 
      "Angiogenesis", "Apoptosis", 
    "Cytokines"
  )
)

dev.off()

pdf 
  2

## Figure 3e,f: Fragmentome Signatures

### Construct signatures from SL-ratios

In [8]:
library(tidyverse)
library(caret)
library(recipes)

lucas_df <- read_csv("../../data/lucas_df_with_pcs.csv") %>%
  mutate(olink_CEA = log2(clinical_CEA + 1)) %>%
  dplyr::select(-starts_with("ratio_pc_"))

# Training features
features_train <- lucas_df %>%
  dplyr::select(id, type, starts_with("ratio_", ignore.case = FALSE))

# Recipe and model training of Mathios et. al. 2021
recipe_seq <- recipe(type ~ ., data = features_train) %>%
  update_role(id, new_role = "ID") %>%
  step_pca(starts_with("ratio_"), prefix = "ratio_pc_", threshold = 0.90) %>%
  step_corr(all_predictors(), threshold = 0.95) %>%
  step_nzv(all_predictors())

set.seed(1234)
model_seq <- train(
  recipe_seq,
  data = features_train,
  method = "glmnet",
  tuneGrid = expand.grid(alpha = 1, lambda = 10^seq(-5, -1, length.out = 100)),
  trControl = trainControl(
    method = "repeatedcv", number = 5, repeats = 10,
    savePredictions = "final", classProbs = TRUE,
    index = createMultiFolds(features_train$type, 5, 10),
    summaryFunction = twoClassSummary
  ),
  metric = "ROC"
)

# Bake PCs and extract loadings
rec_prep <- prep(recipe_seq, training = features_train, retain = TRUE)
baked_lucas <- bake(rec_prep, new_data = lucas_df)

loadings_df <- tidy(rec_prep, number = which(sapply(rec_prep$steps, inherits, "step_pca"))) %>%
  filter(component %in% paste0("PC", 1:11)) %>%
  dplyr::select(component, terms, value) %>%
  pivot_wider(names_from = terms, values_from = value)
loadings_df

Rows: 287 Columns: 958
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr   (96): Patient, id, type, clinical_smokingstatus, QC, patient.type, cli...
dbl  (850): multinucratio, clinical_nlratio, clinical_CRP, clinical_cfdna_co...
lgl    (3): DIAGNOSE_OTHER_PRIMARY, Adaptor Dimer, Date Data Received
date   (9): DATE_1_VISIT_BBH, DATE_BIOPSY, DATE_SCAN_BASELINE, DATE_FEV1, DA...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Loading required namespace: glmnet

Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loaded glmnet 4.1-8



component,ratio_1,ratio_2,ratio_3,ratio_4,ratio_5,ratio_6,ratio_7,ratio_8,ratio_9,⋯,ratio_464,ratio_465,ratio_466,ratio_467,ratio_468,ratio_469,ratio_470,ratio_471,ratio_472,ratio_473
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PC1,0.048038750,0.047013441,0.053124462,0.162708956,0.006119247,0.036205161,0.082127054,0.003564283,0.032599370,⋯,-0.037706713,-3.868317e-02,0.007025112,0.0345754739,0.021807837,0.001101796,0.03666598,0.111698320,0.107737649,0.006211197
PC2,-0.005859589,-0.017480205,-0.001271587,0.037762289,-0.030504256,-0.018869736,-0.015428969,-0.024359563,-0.015305888,⋯,0.032051223,1.535480e-02,0.003763096,0.0007236314,-0.004074744,0.006730539,0.02437359,0.042081735,0.043710679,0.021960364
PC3,-0.063138191,-0.082600797,-0.055112745,0.073753724,-0.099661300,-0.035915671,-0.022846810,-0.062563056,-0.017243432,⋯,0.085044222,6.416326e-03,-0.021869894,-0.0354620166,-0.078479486,-0.092500817,-0.04627191,-0.009492434,0.002719488,-0.102715365
PC4,0.003430403,-0.004481853,0.008365886,-0.008059535,-0.063593779,-0.063633148,-0.032797932,-0.038140202,-0.038309152,⋯,-0.022774499,4.959216e-03,-0.023352198,-0.0098635981,-0.019200105,0.065096503,0.05510924,0.035604163,0.052592893,-0.009954337
PC5,-0.036675498,-0.030059874,-0.020091063,-0.056086721,-0.006289272,-0.008306533,0.002115103,0.009906922,0.018760409,⋯,-0.057898198,-3.069373e-02,-0.032746772,-0.0297078395,-0.034719062,0.059811421,0.05083251,0.007436012,0.019861547,0.023117737
PC6,-0.002139730,0.020756077,0.028302489,0.005574373,-0.007058259,0.004673583,0.058030787,0.051647598,0.044031992,⋯,-0.047060706,4.749845e-03,0.018725244,0.0264349675,-0.014459327,-0.016838970,-0.03147522,-0.094202305,-0.048139335,-0.111415362
PC7,0.100744633,0.081437980,0.029759175,-0.027528005,-0.019984329,-0.055484366,-0.053709441,-0.062502519,-0.088728717,⋯,-0.017937803,-1.932470e-02,-0.020658137,-0.0263971415,-0.001527160,0.089512649,0.07233937,0.083799957,0.051342634,0.099360448
PC8,-0.073639134,-0.069735791,-0.052175552,-0.042893647,-0.015444737,-0.013540707,0.027504771,0.017080138,0.035704861,⋯,-0.058777841,-2.525514e-02,-0.050652630,-0.0744867248,-0.075395759,0.029282560,0.01140899,-0.009740690,0.007469584,-0.060722250
PC9,0.012671672,-0.031736502,-0.032492130,-0.041948296,-0.021762581,-0.020063435,-0.039114941,-0.015944039,0.014799677,⋯,-0.008581059,-9.839851e-05,0.013782404,0.0167481901,0.024298844,0.037385181,0.04182326,0.025914451,0.021532232,0.049021345


In [9]:
## fragmentation profiles - LUCAS (training)
bins5mb_old <- read_csv("../../data/lucas_5mbs_delfi473_2025.csv")

# Get unique mapping of bin → arm
bin_to_arm <- bins5mb_old %>%
  dplyr::select(bin, arm) %>%
  distinct()

Rows: 191456 Columns: 19
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr  (3): id, chr, arm
dbl (16): start, end, bin, gc, map, short, long, short.cor, long.cor, ultras...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [10]:
library(dplyr)
library(tidyr)
library(stringr)

# ---------------------------------------------------------
# 1. Identify all ratio_* columns in the loadings dataframe
# ---------------------------------------------------------
ratio_cols <- grep("^ratio_", names(loadings_df), value = TRUE)

# ---------------------------------------------------------
# 2. Extract bin numbers from ratio column names
#     e.g., "ratio_57" → 57
# ---------------------------------------------------------
bin_numbers <- as.integer(str_replace(ratio_cols, "ratio_", ""))

ratio_lookup <- data.frame(
  ratio_col = ratio_cols,
  bin = bin_numbers
)

# ---------------------------------------------------------
# 3. Merge with bin_to_arm lookup table
#     bin_to_arm must have columns: bin, arm
# ---------------------------------------------------------
ratio_lookup2 <- ratio_lookup %>%
  left_join(bin_to_arm, by = "bin")
# Now contains: ratio_col | bin | arm

# ---------------------------------------------------------
# 4. Pivot loadings_df long and merge annotations
# ---------------------------------------------------------
loadings_long <- loadings_df %>%
  pivot_longer(
    cols = starts_with("ratio_"),
    names_to = "ratio_col",
    values_to = "loading"
  ) %>%
  left_join(ratio_lookup2, by = "ratio_col") %>%
  arrange(component, bin)

# ---------------------------------------------------------
# 5. Inspect
# ---------------------------------------------------------
head(loadings_long)


component,ratio_col,loading,bin,arm
<chr>,<chr>,<dbl>,<dbl>,<chr>
PC1,ratio_1,0.048038750,1,1p
PC1,ratio_2,0.047013441,2,1p
PC1,ratio_3,0.053124462,3,1p
PC1,ratio_4,0.162708956,4,1p
PC1,ratio_5,0.006119247,5,1p
PC1,ratio_6,0.036205161,6,1p


In [11]:
write_csv(loadings_long, "../../data/Frag-PC-loadings-long.csv")

### Plot Heatmap

In [12]:
plot_feature_heatmap <- function(
  df,
  feature_type = c("ratio_pc","zscore","olink","tf","custom"),
  custom_features = NULL,
  control_for_ichor = FALSE,
  clinical_vars = c("clinical_Hgb","clinical_platelets","clinical_nlratio","clinical_survival",
                    "clinical_cfdna_conc","clinical_CRP","clinical_smokingstatus",
                    "clinical_leukocytes","clinical_age","clinical_packyears", 
                    "clinical_bmi","clinical_COPD","clinical_sex"),
  olink_groups = NULL,
  abs_cor = TRUE,
  color_scheme = c("red","blue","purple","green"),
  cor_max = 0.4,
  loadings_df = NULL,
  bin_to_arm   = NULL,
  pc_split_after = 2,
  column_gap_size = 8,
  pc_minor_gap_size = 2,
  top_panel_width_multiplier = 1.0,
  cell_width = 1.2
) {

  feature_type <- match.arg(feature_type)
  color_scheme <- match.arg(color_scheme)

  # ------------------------
  # Olink groups
  # ------------------------
  olink_map <- list(
    "Tumor" = c("olink_CEA","olink_MMP12"), 
    "Metabolic" = c("olink_ADA","olink_ARG1","olink_CAIX","olink_HO_1"),
    "Cytokines" = c("olink_IL2","olink_IL4","olink_IL5","olink_IL6","olink_IL7",
                       "olink_IL8","olink_IL10","olink_IL12","olink_IL13","olink_IL18","olink_IL33",
                       "olink_IL_1_alpha","olink_IL_21","olink_IL_35","olink_IL12RB1"),
    "T-cell" = c("olink_CD4","olink_CD5","olink_CD8A","olink_CD27","olink_CD28","olink_CRTAM"),
    "B-cell" = c("olink_CD40","olink_CD83","olink_LAMP3"),
    "NK" = c("olink_NCR1","olink_KLRD1"),
    "Checkpoint" = c("olink_PD_L1","olink_PD_L2","olink_PDCD1","olink_ICOSLG"),
    "TNF Superfamily" = c("olink_TNF","olink_TNFSF14","olink_TNFRSF4","olink_TNFRSF9",
                          "olink_TNFRSF12A","olink_TNFRSF21","olink_CD40_L","olink_CD70"),
    "Apoptosis" = c("olink_TRAIL","olink_FASLG","olink_TWEAK","olink_CASP_8"),
    "Granzymes" = c("olink_GZMA","olink_GZMB","olink_GZMH"),
    "Chemokines" = c("olink_CXCL1","olink_CXCL5","olink_CXCL9","olink_CXCL10","olink_CXCL11",
                     "olink_CXCL12","olink_CXCL13","olink_CCL3","olink_CCL4","olink_CCL17",
                     "olink_CCL19","olink_CCL20","olink_CCL23","olink_MCP_1","olink_MCP_2",
                     "olink_MCP_3","olink_MCP_4","olink_CX3CL1"),
    "Angiogenesis" = c("olink_VEGFA","olink_VEGFC","olink_VEGFR_2","olink_ANG_1",
                       "olink_ANGPT2","olink_TIE2","olink_PGF","olink_NOS3"),
    "Growth Factors" = c("olink_EGF","olink_FGF2","olink_HGF","olink_PDGF_subunit_B","olink_CSF_1","olink_PTN"),
    "Interferons" = c("olink_IFN_beta","olink_IFN_gamma"),
    "Galectins" = c("olink_Gal_1","olink_Gal_9"),
    "Stress/Cytotoxicity Markers" = c("olink_MIC_A_B","olink_CD244"),
    "Other Proteins" = c("olink_DCN","olink_ADGRG1","olink_LAP_TGF_beta_1", "olink_MMP7")
  )

  # ------------------------------
  # Clinical vars + olink filtering
  # ------------------------------
  if (!is.null(olink_groups)) {
    selected_olinks <- unlist(olink_map[olink_groups])
    clinical_vars <- c(
      clinical_vars[!grepl("^olink_", clinical_vars)],
      selected_olinks
    )
  } else {
    all_olinks <- unlist(olink_map)
    clinical_vars <- c(
      clinical_vars[!grepl("^olink_", clinical_vars)],
      all_olinks
    )
  }

  # ------------------------------
  # Feature columns
  # ------------------------------
  feature_cols <- switch(
    feature_type,
    "ratio_pc" = grep("^ratio_pc_", names(df), value=TRUE),
    "zscore"   = grep("^zscore_",   names(df), value=TRUE),
    "olink"    = grep("^olink_",    names(df), value=TRUE),
    "tf"       = grep("^relcov-.*-rel_cov_profile$", names(df), value=TRUE),
    "custom"   = custom_features
  )
  stopifnot(length(feature_cols) > 0)

  clinical_vars <- clinical_vars[clinical_vars %in% names(df)]
  clinical_vars <- clinical_vars[sapply(df[clinical_vars], is.numeric)]

  if (!control_for_ichor && "ichor" %in% names(df)) {
    clinical_vars <- c(clinical_vars, "ichor")
  }

  # ---------------------------------
  # Compute correlations
  # ---------------------------------
  cor_long <- expand.grid(
    clinical = clinical_vars,
    feature  = feature_cols,
    stringsAsFactors = FALSE
  ) %>%
    mutate(cor = pmap_dbl(list(clinical, feature), function(cvar, fvar) {
      x <- df[[cvar]]
      y <- df[[fvar]]
      if (control_for_ichor) {
        safe_pcor(x, y, df$ichor)
      } else {
        suppressWarnings(cor(x, y, use="pairwise.complete.obs"))
      }
    }))

  if (abs_cor) cor_long$cor <- abs(cor_long$cor)

  cor_mat <- cor_long %>%
    pivot_wider(names_from = feature, values_from = cor) %>%
    column_to_rownames("clinical") %>%
    as.matrix()

  cor_mat[is.na(cor_mat)] <- 0

  # ---------------------------------
  # Row grouping
  # ---------------------------------
  original_rownames <- rownames(cor_mat)

  row_split <- sapply(original_rownames, function(var) {
    if (var %in% c("ichor", "GEMINI_score", "DELFI-TS")) {
      "01_ctDNA"
    } else if (grepl("^clinical_leukocytes$|^clinical_nlratio$", var)) {
      "02_Inflammatory Markers"
    } else if (grepl("^clinical_Hgb$|^clinical_platelets$", var)) {
      "03_Hematologic"
    } else if (grepl("^clinical_age$|^clinical_sex$|^clinical_bmi$", var)) {
      "04_Demographics"
    } else if (grepl("^clinical_smokingstatus$|^clinical_packyears$|^clinical_COPD$", var)) {
      "05_Smoking/Respiratory"
    } else if (grepl("^clinical_CRP$|^clinical_cfdna_conc$", var)) {
      "06_cfDNA/CRP"
    } else if (grepl("^clinical_survival$", var)) {
      "99_Survival"
    } else if (grepl("^clinical_", var)) {
      "07_Other Clinical"
    } else if (grepl("^olink_", var)) {
      for (group_name in names(olink_map)) {
        if (var %in% olink_map[[group_name]]) {
          group_num <- match(group_name, names(olink_map)) + 9
          return(sprintf("%02d_%s", group_num, group_name))
        }
      }
      "89_Other Proteins"
    } else {
      "07_Other Clinical"
    }
  })

  row_info <- data.frame(
    original_name = original_rownames,
    category = row_split,
    stringsAsFactors = FALSE
  )
  row_info <- row_info[order(row_info$category), ]

  cor_mat <- cor_mat[row_info$original_name, , drop = FALSE]
  row_split <- row_info$category

  # ---------------------------------
  # Clean row labels
  # ---------------------------------
  rownames(cor_mat) <- gsub("^clinical_", "", rownames(cor_mat))
  rownames(cor_mat) <- tools::toTitleCase(rownames(cor_mat))
  rownames(cor_mat) <- gsub("^Ichor$", "ichorCNA        ", rownames(cor_mat))
  rownames(cor_mat) <- gsub("GEMINI_score", "GEMINI        ", rownames(cor_mat))
  rownames(cor_mat) <- gsub("Cfdna_conc", "cfDNA Conc.", rownames(cor_mat))
  rownames(cor_mat) <- gsub("Nlratio", "Neutrophiles", rownames(cor_mat))
  rownames(cor_mat) <- gsub("^olink_", "", rownames(cor_mat))

  # ---------------------------------
  # Column name cleaning & ordering + column splitting
  # ---------------------------------
  column_split <- NULL
  
  if (feature_type == "ratio_pc") {
    colnames(cor_mat) <- gsub("^ratio_pc_0*", "", colnames(cor_mat))
    pc_nums <- as.numeric(colnames(cor_mat))
    ord <- order(pc_nums)
    colnames(cor_mat) <- as.character(pc_nums)
    cor_mat <- cor_mat[, ord, drop = FALSE]
    
    # Create column split for each individual PC
    # This allows gaps between all PCs
    pc_numbers <- as.numeric(colnames(cor_mat))
    
    # Create split labels: "A1", "A2" for PCs before split, "B3", "B4", etc. for PCs after
    column_split <- sapply(pc_numbers, function(pc) {
      if (pc <= pc_split_after) {
        paste0("A", sprintf("%02d", pc))
      } else {
        paste0("B", sprintf("%02d", pc))
      }
    })
    column_split <- factor(column_split, levels = unique(column_split))
    
  } else if (feature_type == "zscore") {
    colnames(cor_mat) <- gsub("^zscore_", "", colnames(cor_mat))
    ordered_cols <- order_chrom_arms(colnames(cor_mat))
    cor_mat <- cor_mat[, ordered_cols, drop = FALSE]
  }

  # ---------------------------------
  # Row split factor and group labels
  # ---------------------------------
  row_split <- factor(row_split, levels = unique(row_split))
  clean_group_labels <- gsub("^[0-9]+_", "", levels(row_split))
  clean_group_labels <- paste0("     ", clean_group_labels)

  # ---------------------------------
  # Color scale
  # ---------------------------------
  mid_point <- cor_max / 2
  col_fun <- switch(
    color_scheme,
    "red"    = colorRamp2(c(0, mid_point, cor_max), c("white","orange","red")),
    "blue"   = colorRamp2(c(0, mid_point, cor_max), c("white","lightblue","darkblue")),
    "purple" = colorRamp2(c(0, mid_point, cor_max), c("white","plum","purple4")),
    "green"  = colorRamp2(c(0, mid_point, cor_max), c("white","lightgreen","darkgreen"))
  )

  legend_breaks <- seq(0, cor_max, length.out = 5)
  legend_labels <- sprintf("%.2f", legend_breaks)

  # ===================================================================
  # Prepare data for top panel (ratio_pc only)
  # ===================================================================
  load_sub2 <- NULL
  arm_map2 <- NULL
  total_xspan <- NULL
  pc_nums_in_heatmap <- NULL
  n_pcs_total <- NULL
  panel_height <- NULL

  if (feature_type == "ratio_pc") {

    if (is.null(loadings_df) || is.null(bin_to_arm)) {
      stop("For feature_type='ratio_pc', please provide loadings_df and bin_to_arm.")
    }

    ratio_cols_ld <- grep("^ratio_", names(loadings_df), value = TRUE)
    bin_numbers <- as.integer(sub("^ratio_", "", ratio_cols_ld))
    
    ratio_lookup <- data.frame(
      ratio_col = ratio_cols_ld, 
      bin = bin_numbers,
      stringsAsFactors = FALSE
    )

    ratio_lookup2 <- ratio_lookup %>%
      left_join(bin_to_arm, by = "bin")

    loadings_long <- loadings_df %>%
      pivot_longer(
        cols = all_of(ratio_cols_ld),
        names_to = "ratio_col",
        values_to = "loading"
      ) %>%
      left_join(ratio_lookup2, by = "ratio_col") %>%
      filter(!is.na(arm), !is.na(bin))

    pc_nums_in_heatmap <- colnames(cor_mat)
    pc_labels_in_heatmap <- paste0("PC", pc_nums_in_heatmap)

    load_sub <- loadings_long %>%
      filter(component %in% pc_labels_in_heatmap)

    arm_map2 <- load_sub %>%
      dplyr::select(arm, bin) %>%
      distinct() %>%
      group_by(arm) %>%
      summarise(
        min_bin = min(bin),
        max_bin = max(bin),
        .groups = "drop"
      ) %>%
      arrange(
        as.numeric(sub("[pq].*$", "", arm)),
        !grepl("p$", arm),
        arm
      ) %>%
      mutate(
        arm_length = max_bin - min_bin + 1,
        start_x = cumsum(lag(arm_length + 3, default = 0)),
        end_x = start_x + arm_length,
        mid_x = (start_x + end_x) / 2
      )

    total_xspan <- max(arm_map2$end_x)

    load_sub2 <- load_sub %>%
      left_join(arm_map2 %>% dplyr::select(arm, min_bin, start_x), by = "arm") %>%
      mutate(x_coord = start_x + (bin - min_bin))

    n_pcs_total <- length(pc_labels_in_heatmap)
    track_height <- 2.5
    panel_height <- n_pcs_total * track_height
  }

  # ---------------------------------
  # HEATMAP (without top annotation for ratio_pc)
  # ---------------------------------
  
  bottom_title_label <- if (feature_type == "zscore") {
    "Z-Score CN-Arms"
  } else if (feature_type == "ratio_pc") {
    "Fragmentome Ratio PCs"
  } else {
    "Features"
  }

  # Calculate heatmap width including all gaps
  if (feature_type == "ratio_pc") {
    n_pcs <- ncol(cor_mat)
    n_minor_gaps <- n_pcs - 1  # gaps between all PCs
    # The big gap replaces one minor gap at the split point
    total_gap_cm <- (column_gap_size / 10) + ((n_minor_gaps - 1) * (pc_minor_gap_size / 10))
    heatmap_width_cm <- n_pcs * cell_width + total_gap_cm
  } else {
    heatmap_width_cm <- ncol(cor_mat) * cell_width + (column_gap_size / 10)
  }
  
  # Create column gap vector for different gap sizes
  if (feature_type == "ratio_pc") {
    n_splits <- length(unique(column_split))
    # Create gap sizes: minor gaps between all, but big gap after pc_split_after
    column_gap_vector <- rep(unit(pc_minor_gap_size, "mm"), n_splits - 1)
    # Replace the gap after pc_split_after with the big gap
    column_gap_vector[pc_split_after] <- unit(column_gap_size, "mm")
  } else {
    column_gap_vector <- unit(column_gap_size, "mm")
  }
  
  ht <- Heatmap(
    cor_mat,
    name = if (control_for_ichor) "Partial Correlation" else "Correlation",
    col = col_fun,
    cluster_rows = TRUE,
    cluster_row_slices = FALSE,
    cluster_columns = if (feature_type == "ratio_pc") FALSE else TRUE,
    show_column_dend = if (feature_type == "ratio_pc") FALSE else TRUE,
    column_dend_side = "top",
    column_dend_height = unit(2, "cm"),
    
    # Row splitting
    row_split = row_split,
    row_gap = unit(8, "mm"),
    
    # Column splitting (for ratio_pc mode - now each PC is its own slice)
    column_split = column_split,
    column_gap = column_gap_vector,
    cluster_column_slices = FALSE,

    # Row group labels (on right)
    row_title = clean_group_labels,
    row_title_side = "right",
    row_title_rot = 270,
    row_title_gp = gpar(fontsize = 20),

    # Column titles - hide for split columns
    column_title = if (!is.null(column_split)) NULL else bottom_title_label,
    column_title_side = "bottom",
    column_title_gp = gpar(fontsize = 24, fontface = "bold"),

    row_names_side = "right",
    column_names_side = "bottom",
    row_names_gp = gpar(fontsize = 20),
    column_names_gp = gpar(fontsize = 20),
    column_names_rot = 0,
    column_names_max_height = unit(10, "cm"),

    rect_gp = gpar(col = "black", lwd = 0.5),
    width = unit(ncol(cor_mat) * cell_width, "cm"),
    height = unit(nrow(cor_mat), "cm"),

    heatmap_legend_param = list(
      title = if (control_for_ichor) "Partial Corr." else "|Corr.|",
      legend_direction = "horizontal",
      title_position = "topcenter",
      at = legend_breaks,
      labels = legend_labels,
      border = "black",
      legend_gp = gpar(col = "black", lwd = 2),
      legend_height = unit(5, "cm"),
      grid_height = unit(8, "mm"),
      grid_width = unit(6, "mm"),
      labels_gp = gpar(fontsize = 15),
      title_gp = gpar(fontsize = 20)
    )
  )

  options(repr.plot.width = 25, repr.plot.height = 50, repr.plot.res = 300)

  # ===================================================================
  # DRAW: Separate top panel and heatmap for ratio_pc
  # ===================================================================
  
  if (feature_type == "ratio_pc" && !is.null(load_sub2)) {
    
    # Calculate widths
    top_panel_width_cm <- heatmap_width_cm * top_panel_width_multiplier
    
    # Create the layout with separate viewports
    grid.newpage()
    
    # Define layout: top panel + spacing + heatmap
    pushViewport(viewport(layout = grid.layout(
      nrow = 3,
      ncol = 1,
      heights = unit(c(panel_height, 2, nrow(cor_mat) + 10), c("cm", "cm", "cm"))
    )))
    
    # ----- TOP PANEL (row 1) -----
    pushViewport(viewport(layout.pos.row = 1, layout.pos.col = 1))
    
    # Center the top panel and make it wider
    pushViewport(viewport(
      x = 0.5,
      y = 0.5,
      width = unit(top_panel_width_cm, "cm"),
      height = unit(panel_height, "cm"),
      xscale = c(-10, total_xspan + 20),
      yscale = c(0.5, n_pcs_total + 0.5)
    ))
    
    # Draw the PC tracks
    slice_pc_labels <- paste0("PC", pc_nums_in_heatmap)
    plot_data <- load_sub2
    
    for (i in seq_along(slice_pc_labels)) {
      pc_label <- slice_pc_labels[i]
      
      pc_data <- plot_data[plot_data$component == pc_label, ]
      pc_data <- pc_data[order(pc_data$x_coord), ]
      
      if (nrow(pc_data) > 0) {
        y_pos <- n_pcs_total - i + 1
        
        # Baseline
        grid.lines(
          x = unit(c(0, total_xspan), "native"),
          y = unit(c(y_pos, y_pos), "native"),
          gp = gpar(col = "grey30", lwd = 1, lty = 1)
        )
        
        # Draw by arm
        for (arm_idx in seq_len(nrow(arm_map2))) {
          current_arm <- arm_map2$arm[arm_idx]
          arm_data <- pc_data[pc_data$arm == current_arm, ]
          
          if (nrow(arm_data) > 0) {
            arm_data <- arm_data[order(arm_data$x_coord), ]
            
            x_vals <- arm_data$x_coord
            loading_vals <- arm_data$loading
            
            max_abs_loading <- max(abs(loading_vals), na.rm = TRUE)
            if (max_abs_loading > 0) {
              y_offset <- (loading_vals / max_abs_loading) * 0.4
            } else {
              y_offset <- rep(0, length(loading_vals))
            }
            
            y_vals <- y_pos + y_offset
            
            if (length(x_vals) > 1) {
              grid.polyline(
                x = unit(x_vals, "native"),
                y = unit(y_vals, "native"),
                gp = gpar(col = "purple3", lwd = 1.25)
              )
            }
          }
        }
        
        # PC label
        grid.text(
          sub("^PC", "", pc_label),
          x = unit(-5, "native"),
          y = unit(y_pos, "native"),
          just = "right",
          gp = gpar(fontsize = 30)
        )
      }
    }
    
    # Arm tick marks
    for (i in seq_len(nrow(arm_map2))) {
      grid.lines(
        x = unit(c(arm_map2$start_x[i], arm_map2$end_x[i]), "native"),
        y = unit(0.55, "native"),
        gp = gpar(col = "black", lwd = 1.5)
      )
    }
    
    # Arm labels
    for (i in seq_len(nrow(arm_map2))) {
      grid.text(
        arm_map2$arm[i],
        x = unit(arm_map2$mid_x[i], "native"),
        y = unit(0.35, "native"),
        just = "top",
        rot = 45,
        gp = gpar(fontsize = 18)
      )
    }
    
    popViewport()  # inner viewport
    
    # Add "Genome-wide PC Loadings" label on left
    grid.text(
      "Genome-wide PC Loadings",
       x = unit(0.5, "npc") - unit(top_panel_width_cm / 2, "cm") - unit(2, "cm"),
      y = unit(0.5, "npc"),
      just = "center",
      rot = 90,
      gp = gpar(fontsize = 30)
    )
    
    popViewport()  # row 1 viewport
    
    # ----- HEATMAP (row 3) -----
    pushViewport(viewport(layout.pos.row = 3, layout.pos.col = 1))
    
    # Create a left-aligned viewport for the heatmap
    # x offset accounts for legend space on the left
    pushViewport(viewport(
      x = unit(-12, "cm"),
      just = "left"
    ))
    
    draw(ht, 
         heatmap_legend_side = "left",
         newpage = FALSE)
    
    # Draw centered x-axis title manually
    grid.text(
      bottom_title_label,
      x = unit(0.5, "npc"),
      y = unit(3.5, "cm"),  # Adjust this to position below column names
      just = "center",
      gp = gpar(fontsize = 24)
    )
    
    popViewport()  # inner left-aligned viewport
    popViewport()  # row 3 viewport
    popViewport()  # main layout viewport
    
  } else {
    # Non-ratio_pc modes: draw normally
    if (!is.null(column_split)) {
      draw(ht, 
           heatmap_legend_side = "left", 
           padding = unit(c(10, 2, 10, 2), "mm"),
           column_title = bottom_title_label,
           column_title_side = "bottom",
           column_title_gp = gpar(fontsize = 24))
    } else {
      draw(ht, 
           heatmap_legend_side = "left", 
           padding = unit(c(10, 2, 10, 2), "mm"))
    }
  }
}

In [13]:
plot_feature_heatmap(
  df,
  feature_type = "ratio_pc",
  color_scheme = "purple", top_panel_width_multiplier = 3.0,
  cor_max = 0.5,
    pc_split_after = 0,
  loadings_df = loadings_df,
  bin_to_arm = bin_to_arm,
  clinical_vars = c("GEMINI_score"),
  olink_groups = c(
    "Tumor", "T-cell", "Checkpoint",
    "Granzymes", "Angiogenesis", "Apoptosis", "B-cell", "NK",
    "Cytokines"
  ),
)

In [14]:
save_heatmap_data <- function(
  df,
  feature_type = c("ratio_pc","zscore","olink","tf","custom"),
  custom_features = NULL,
  control_for_ichor = FALSE,
  clinical_vars = c("clinical_Hgb","clinical_platelets","clinical_nlratio","clinical_survival",
                    "clinical_cfdna_conc","clinical_CRP","clinical_smokingstatus",
                    "clinical_leukocytes","clinical_age","clinical_packyears", 
                    "clinical_bmi","clinical_COPD","clinical_sex"),
  olink_groups = NULL,
  abs_cor = TRUE
) {
  
  feature_type <- match.arg(feature_type)
  
  olink_map <- list(
    "Tumor" = c("olink_CEA","olink_MMP12"), 
    "Metabolic" = c("olink_ADA","olink_ARG1","olink_CAIX","olink_HO_1"),
    "Cytokines" = c("olink_IL2","olink_IL4","olink_IL5","olink_IL6","olink_IL7",
                       "olink_IL8","olink_IL10","olink_IL12","olink_IL13","olink_IL18","olink_IL33",
                       "olink_IL_1_alpha","olink_IL_21","olink_IL_35","olink_IL12RB1"),
    "T-cell" = c("olink_CD4","olink_CD5","olink_CD8A","olink_CD27","olink_CD28","olink_CRTAM"),
    "B-cell" = c("olink_CD40","olink_CD83","olink_LAMP3"),
    "NK Cell" = c("olink_NCR1","olink_KLRD1"),
    "Checkpoint" = c("olink_PD_L1","olink_PD_L2","olink_PDCD1","olink_ICOSLG"),
    "TNF Superfamily" = c("olink_TNF","olink_TNFSF14","olink_TNFRSF4","olink_TNFRSF9",
                          "olink_TNFRSF12A","olink_TNFRSF21","olink_CD40_L","olink_CD70"),
    "Apoptosis" = c("olink_TRAIL","olink_FASLG","olink_TWEAK","olink_CASP_8"),
    "Granzymes" = c("olink_GZMA","olink_GZMB","olink_GZMH"),
    "Chemokines" = c("olink_CXCL1","olink_CXCL5","olink_CXCL9","olink_CXCL10","olink_CXCL11",
                     "olink_CXCL12","olink_CXCL13","olink_CCL3","olink_CCL4","olink_CCL17",
                     "olink_CCL19","olink_CCL20","olink_CCL23","olink_MCP_1","olink_MCP_2",
                     "olink_MCP_3","olink_MCP_4","olink_CX3CL1"),
    "Angiogenesis" = c("olink_VEGFA","olink_VEGFC","olink_VEGFR_2","olink_ANG_1",
                       "olink_ANGPT2","olink_TIE2","olink_PGF","olink_NOS3"),
    "Growth Factors" = c("olink_EGF","olink_FGF2","olink_HGF","olink_PDGF_subunit_B","olink_CSF_1","olink_PTN"),
    "Interferons" = c("olink_IFN_beta","olink_IFN_gamma"),
    "Galectins" = c("olink_Gal_1","olink_Gal_9"),
    "Stress/Cytotoxicity Markers" = c("olink_MIC_A_B","olink_CD244"),
    "Other Proteins" = c("olink_DCN","olink_ADGRG1","olink_LAP_TGF_beta_1","olink_MMP7")
  )
  
  # Filter olink vars
  if (!is.null(olink_groups)) {
    selected_olinks <- unlist(olink_map[olink_groups])
    clinical_vars <- c(clinical_vars[!grepl("^olink_", clinical_vars)], selected_olinks)
  } else {
    clinical_vars <- c(clinical_vars[!grepl("^olink_", clinical_vars)], unlist(olink_map))
  }
  
  # Feature columns
  feature_cols <- switch(
    feature_type,
    "ratio_pc" = grep("^ratio_pc_", names(df), value = TRUE),
    "zscore" = grep("^zscore_", names(df), value = TRUE),
    "olink" = grep("^olink_", names(df), value = TRUE),
    "tf" = grep("^relcov-.*-rel_cov_profile$", names(df), value = TRUE),
    "custom" = custom_features
  )
  
  clinical_vars <- clinical_vars[clinical_vars %in% names(df)]
  clinical_vars <- clinical_vars[sapply(df[clinical_vars], is.numeric)]
  
  if (!control_for_ichor && "ichor" %in% names(df)) {
    clinical_vars <- c(clinical_vars, "ichor")
  }
  
  # Compute correlations
  cor_long <- expand.grid(clinical = clinical_vars, feature = feature_cols, stringsAsFactors = FALSE) %>%
    mutate(cor = purrr::pmap_dbl(list(clinical, feature), function(cvar, fvar) {
      x <- df[[cvar]]
      y <- df[[fvar]]
      if (control_for_ichor && "ichor" %in% names(df)) {
        tryCatch({
          data <- data.frame(x = x, y = y, z = df$ichor)[complete.cases(data.frame(x = x, y = y, z = df$ichor)), ]
          if (nrow(data) < 3) return(NA_real_)
          ppcor::pcor.test(data$x, data$y, data$z)$estimate
        }, error = function(e) NA_real_)
      } else {
        suppressWarnings(cor(x, y, use = "pairwise.complete.obs"))
      }
    }))
  
  if (abs_cor) cor_long$cor <- abs(cor_long$cor)
  
  cor_mat <- cor_long %>%
    pivot_wider(names_from = feature, values_from = cor) %>%
    column_to_rownames("clinical")
  
  list(
    cor_matrix = as.data.frame(cor_mat),
    cor_long = cor_long
  )
}

# Use it
heatmap_data <- save_heatmap_data(
  df,
  feature_type = "ratio_pc",
    clinical_vars = c("GEMINI_score"),
  olink_groups = c("Tumor", "T-cell", "B-cell", "NK Cell",
                   "Checkpoint", "Granzymes", "Angiogenesis", "Apoptosis", "Cytokines")
)

        
# Save
write.csv(heatmap_data$cor_matrix, "../../data/Frag_Signature_heatmap_correlation_matrix.csv")


In [15]:
png("../../outputs/Fig3/Fig-Ratio-PC_tracks_heatmap.png", width = 25, height = 50, units = "in", res = 600)

plot_feature_heatmap(
  df,
  feature_type = "ratio_pc",
  color_scheme = "purple", top_panel_width_multiplier = 3.0,
  cor_max = 0.5,
    pc_split_after = 0,
    loadings_df=loadings_df,
    bin_to_arm=bin_to_arm,
    clinical_vars=c("GEMINI_score"),
  olink_groups = c(
    "Tumor", "T-cell","Checkpoint",
    "Granzymes", "Angiogenesis","Apoptosis", "B-cell", "NK",
    "Cytokines"
  )
)

dev.off()

pdf 
  2

In [ ]:
# Done #